# Ad-moment dataset: Bronze → Silver → human anchor

This is the **preprocessing** notebook. It does not train anything and does not need a GPU.
Run it on a machine that has the repo and the tracked JSONL (Atlas, or the laptop with the clone).
Then zip `outputs/{silver,anchor}` and take that zip to Colab / Kaggle for [`ad_moment_scorer.ipynb`](ad_moment_scorer.ipynb).

The parsers live in the two scripts next to this notebook. This file only runs them and prints the checks a later session should not re-derive.

| Zone | Path | Grain |
|---|---|---|
| Bronze | `src/project/logs/tracked/{lab,crowd,beta}/` + `logs/production/` | JSONL events |
| Silver | `outputs/silver/turns.csv` | one row per user turn (1,479) |
| Silver | `outputs/silver/conversations.csv` | one row per condition block (361) |
| Anchor | `outputs/anchor/human_anchor.csv` | 216 advertised conversations |

`source_set`: `primary` (N=54, may carry human labels), `beta` (unlabeled text), `unlabeled` (unfinished / crowdfail / production leftovers). `synthetic` is dropped.

Presentation \(\lambda\) is stored on the row for residualising the human label. It is **not** a model input.

In [ ]:
from pathlib import Path
import json, hashlib, sys
import pandas as pd

HERE = Path.cwd()
if not (HERE / "build_policy_silver.py").exists():
    candidates = [
        Path("analysis/policy"),
        Path("/home/wtroi/MasterThesis-RAG-RecSys/analysis/policy"),
    ]
    HERE = next(p.resolve() for p in candidates if (p / "build_policy_silver.py").exists())
    import os
    os.chdir(HERE)
sys.path.insert(0, str(HERE))
print("cwd", HERE.resolve())

In [ ]:
import build_policy_silver as silver

result = silver.build()
out = HERE / "outputs" / "silver"
out.mkdir(parents=True, exist_ok=True)
silver.write_csv(out / "turns.csv", result["turns"])
silver.write_csv(out / "conversations.csv", result["conversations"])
report = result["report"]
report["files"] = {
    "turns.csv": silver.file_hash(out / "turns.csv"),
    "conversations.csv": silver.file_hash(out / "conversations.csv"),
}
(out / "build_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report["summary"], indent=2))
print("sessions:", report["sessions"])
if report.get("skipped"):
    print("skipped", len(report["skipped"]), "(see build_report.json)")

In [ ]:
import build_human_anchor as anchor_mod

anchor_df, anchor_report = anchor_mod.build(HERE / "outputs" / "silver")
aout = HERE / "outputs" / "anchor"
aout.mkdir(parents=True, exist_ok=True)
path = aout / "human_anchor.csv"
anchor_df.to_csv(path, index=False)
anchor_report["sha256_16"] = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
(aout / "anchor_report.json").write_text(json.dumps(anchor_report, indent=2))
keep = {k: v for k, v in anchor_report.items() if k != "composites"}
print(json.dumps(keep, indent=2))
print("\ncomposites are provisional until the Goal 1 freeze; rebuild this cell after that freeze.")

In [ ]:
turns = pd.read_csv(HERE / "outputs" / "silver" / "turns.csv")
conv = pd.read_csv(HERE / "outputs" / "silver" / "conversations.csv")
anchor = pd.read_csv(HERE / "outputs" / "anchor" / "human_anchor.csv")

print("turns", turns.shape, turns.source_set.value_counts().to_dict())
print("primary participants", turns[turns.source_set == "primary"].participant_key.nunique())
print("conversations", conv.shape, "primary ads", int(((conv.source_set == "primary") & (conv.condition != "no_ads")).sum()))
print("anchor", anchor.shape, "good_moment_rate", round(anchor.good_moment_human.mean(), 3))
print("UX retention by timing\n", anchor.groupby("timing").ux_retention.mean().round(3))
print("UX retention by presentation (nuisance, residualised out of the label)\n",
      anchor.groupby("presentation").ux_retention.mean().round(3))
assert turns[turns.source_set == "primary"].shape[0] == 1080
assert anchor.shape[0] == 216
assert set(anchor.condition) <= {"inline_early", "inline_late", "block_early", "block_late"}
print("ok")

## Take this zip to Colab / Kaggle

Do **not** include `outputs/notebook_runs/` (weights, judge caches). The trainer only needs Silver + anchor.

```bash
cd analysis/policy
zip -r ad_moment_tables.zip outputs/silver outputs/anchor
```

On Colab: upload that zip when the trainer asks, or put it at `MyDrive/ad_moment/outputs` unpacked as `silver/` + `anchor/`.